# Clinical Insulin ML Pipeline — Reprocessing + Training

This notebook runs the full end-to-end **clinical insulin regression** pipeline:

- Loads the SmartSensor CSV
- Reprocesses / feature engineers / splits (group-wise by patient)
- Trains multiple models and selects the best by **test RMSE**
- Saves:
  - **Best model bundle** for the backend (`outputs/best_model/inference_bundle.joblib`)
  - **Evaluation metrics** CSVs (`outputs/clinical_insulin_pipeline/latest/evaluation/`)
  - **Visualizations** (`outputs/clinical_insulin_pipeline/latest/models/`)

## What the app will use
The FastAPI backend loads:
- `outputs/best_model/inference_bundle.joblib`

So after this notebook finishes, the app can immediately serve `/api/recommend`.


In [46]:
from __future__ import annotations

import sys
from pathlib import Path


def _detect_backend_src(start: Path) -> Path:
    """Find backend/src either in this project or one level down."""
    p = start.resolve()
    while True:
        # Case A: notebook started inside Clinical-Insulin-Recommendation/
        cand = p / "backend" / "src"
        if cand.is_dir():
            return cand

        # Case B: notebook started from a parent workspace (e.g. E:\Glucosense app)
        cand2 = p / "Clinical-Insulin-Recommendation" / "backend" / "src"
        if cand2.is_dir():
            return cand2

        if p == p.parent:
            break
        p = p.parent

    raise RuntimeError(
        "Cannot locate backend/src. Start Jupyter in the project folder, "
        "or ensure 'Clinical-Insulin-Recommendation/backend/src' exists."
    )


SRC = _detect_backend_src(Path.cwd())
# repo root is the parent of backend/
repo_root = SRC.parent.parent

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Detected repo root:", repo_root)
print("SRC:", SRC)
print("Python:", sys.version.split()[0])


Detected repo root: E:\Glucosense app\Clinical-Insulin-Recommendation
SRC: E:\Glucosense app\Clinical-Insulin-Recommendation\backend\src
Python: 3.14.3


In [47]:
from clinical_insulin_pipeline.training import run_training
from clinical_insulin_pipeline.config import DEFAULT_DATA_CSV, repo_root_from_here
from clinical_insulin_pipeline.data.dataset import load_raw_csv

repo = repo_root_from_here()
print("Pipeline repo root:", repo)

csv_path = (repo / DEFAULT_DATA_CSV).resolve()
print("Training CSV:", csv_path)
print("CSV exists:", csv_path.is_file())

# Optional knobs
SKIP_LEARNING_CURVE = False
SKIP_SHAP = True  # SHAP can be heavy / optional on Windows


Pipeline repo root: E:\Glucosense app\Clinical-Insulin-Recommendation
Training CSV: E:\Glucosense app\Clinical-Insulin-Recommendation\data\SmartSensor_DiabetesMonitoring.csv
CSV exists: True


## Exploratory Data Analysis (EDA)

This section explores the SmartSensor dataset so we can understand:
- global structure and missingness
- target distribution and skew
- timestamp patterns and seasonality
- correlations between numeric features
- categorical distributions for non-numeric fields

In [48]:
from clinical_insulin_pipeline.data.dataset import load_raw_csv
try:
    from clinical_insulin_pipeline.data.features import (
        add_cyclical_time_features,
        add_time_series_features,
        feature_columns_after_engineering,
    )
except ImportError:
    def add_time_series_features(df):
        df = df.copy()
        if "Timestamp" not in df.columns:
            return df
        ts = pd.to_datetime(df["Timestamp"], errors="coerce")
        df["hour"] = ts.dt.hour
        df["day_of_week"] = ts.dt.day_name()
        df["month"] = ts.dt.month_name()
        return df

    def add_cyclical_time_features(df):
        df = df.copy()
        if "hour" in df.columns:
            df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
            df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        if "month" in df.columns:
            month_num = df["month"].map(
                {
                    "January": 1,
                    "February": 2,
                    "March": 3,
                    "April": 4,
                    "May": 5,
                    "June": 6,
                    "July": 7,
                    "August": 8,
                    "September": 9,
                    "October": 10,
                    "November": 11,
                    "December": 12,
                }
            )
            df["month_sin"] = np.sin(2 * np.pi * month_num / 12)
            df["month_cos"] = np.cos(2 * np.pi * month_num / 12)
        if "day_of_week" in df.columns:
            dow_num = df["day_of_week"].map(
                {
                    "Monday": 0,
                    "Tuesday": 1,
                    "Wednesday": 2,
                    "Thursday": 3,
                    "Friday": 4,
                    "Saturday": 5,
                    "Sunday": 6,
                }
            )
            df["dow_sin"] = np.sin(2 * np.pi * dow_num / 7)
            df["dow_cos"] = np.cos(2 * np.pi * dow_num / 7)
        return df

    def feature_columns_after_engineering():
        return [
            "hour_sin",
            "hour_cos",
            "month_sin",
            "month_cos",
            "dow_sin",
            "dow_cos",
            "Glucose_Level",
            "Heart_Rate",
            "Activity_Level",
            "Calories_Burned",
            "Sleep_Duration",
            "Step_Count",
            "Medication_Intake",
            "Diet_Quality_Score",
            "Stress_Level",
            "BMI",
            "HbA1c",
            "Blood_Pressure_Systolic",
            "Blood_Pressure_Diastolic",
            "glycemic_stress_index",
            "pulse_pressure",
            "activity_volume",
        ]
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from clinical_insulin_pipeline.config import TARGET_COL
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

raw_df = load_raw_csv(csv_path)
print("Dataset shape:", raw_df.shape)
print("Total rows:", len(raw_df))
print("Unique patients:", raw_df["Patient_ID"].nunique())
print("Target rows with zero Insulin_Dose:", (raw_df[TARGET_COL] == 0).sum())
print("Target unique values:", raw_df[TARGET_COL].nunique())
print("Dataset size verdict:", "Good for ensemble models" if len(raw_df) >= 1000 else "Small dataset, simpler models recommended")

display(raw_df.head())

print("\nMissing values per column:")
display(raw_df.isna().sum().sort_values(ascending=False))

print("\nNumeric summary statistics:")
display(raw_df.describe(include="number").T)

print("\nCategorical summary statistics:")
display(raw_df.describe(include=["object"]).T)

if "Timestamp" in raw_df.columns:
    ts = pd.to_datetime(raw_df["Timestamp"], errors="coerce")
    print("\nTimestamp range:", ts.min(), "to", ts.max())
    print("Missing timestamps:", int(ts.isna().sum()))
    if ts.notna().any():
        raw_df["hour"] = ts.dt.hour
        raw_df["day_of_week"] = ts.dt.day_name()
        raw_df["month"] = ts.dt.month_name()

        fig, axs = plt.subplots(1, 3, figsize=(18, 4))
        sns.countplot(x="hour", data=raw_df, ax=axs[0])
        sns.countplot(
            x="day_of_week",
            data=raw_df,
            order=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
            ax=axs[1],
        )
        sns.countplot(
            x="month",
            data=raw_df,
            order=[
                "January",
                "February",
                "March",
                "April",
                "May",
                "June",
                "July",
                "August",
                "September",
                "October",
                "November",
                "December",
            ],
            ax=axs[2],
        )
        for ax in axs:
            ax.tick_params(axis="x", rotation=45)
        fig.suptitle("Timestamp distribution by hour / day / month")
        plt.tight_layout()
        plt.show()

numeric_cols = raw_df.select_dtypes(include=["number"]).columns.tolist()
if numeric_cols:
    print("\nNumeric feature distributions:")
    raw_df[numeric_cols].hist(figsize=(16, 12), bins=20)
    plt.tight_layout()
    plt.show()

if TARGET_COL in raw_df.columns:
    print(f"\nTarget ({TARGET_COL}) distribution and skew:")
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(raw_df[TARGET_COL].dropna(), kde=True, ax=axs[0])
    axs[0].set_title(f"{TARGET_COL} histogram")
    sns.boxplot(x=raw_df[TARGET_COL], ax=axs[1])
    axs[1].set_title(f"{TARGET_COL} boxplot")
    plt.tight_layout()
    plt.show()
    print("Skewness:", float(raw_df[TARGET_COL].skew()))
    print("Kurtosis:", float(raw_df[TARGET_COL].kurtosis()))

if numeric_cols:
    corr = raw_df[numeric_cols].corr()
    print("\nNumeric correlation matrix:")
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation matrix for numeric features")
    plt.show()

object_cols = raw_df.select_dtypes(include=["object"]).columns.tolist()
if object_cols:
    print("\nCategorical value counts (top categories):")
    for col in object_cols:
        if col == "Timestamp":
            continue
        unique_count = raw_df[col].nunique(dropna=False)
        if unique_count <= 20:
            print(f"\n{col} ({unique_count} unique values)")
            display(raw_df[col].value_counts(dropna=False).head(20))

print("\nBaseline linear regression check:")

raw_df = add_time_series_features(raw_df)
raw_df = add_cyclical_time_features(raw_df)
feature_cols = feature_columns_after_engineering()
missing_cols = [c for c in feature_cols if c not in raw_df.columns]
if missing_cols:
    print("Skipping missing engineered features:", missing_cols)

feature_cols = [c for c in feature_cols if c in raw_df.columns]
X = raw_df[feature_cols].copy()
if "time_of_day_category" in X.columns:
    X["time_of_day_category"] = X["time_of_day_category"].astype("category").cat.codes
elif "hour" in raw_df.columns:
    X["time_of_day_category"] = pd.cut(
        raw_df["hour"],
        bins=[-1, 6, 12, 18, 24],
        labels=["night", "morning", "afternoon", "evening"],
        right=False,
    ).astype("category").cat.codes

y = raw_df[TARGET_COL].astype(float)
groups = raw_df["Patient_ID"]

split = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
train_idx, test_idx = next(split.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

baseline_pipe = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LinearRegression()),
    ]
)
X_train_num = baseline_pipe.named_steps["impute"].fit_transform(X_train)
X_train_num = baseline_pipe.named_steps["scaler"].fit_transform(X_train_num)
print("Baseline train rows:", len(X_train))
print("Baseline test rows:", len(X_test))

lr = LinearRegression()
lr.fit(X_train_num, y_train)
X_test_num = baseline_pipe.named_steps["impute"].transform(X_test)
X_test_num = baseline_pipe.named_steps["scaler"].transform(X_test_num)

y_pred = lr.predict(X_test_num)
print("Baseline Linear Regression R2:", r2_score(y_test, y_pred))
print("Baseline MAE:", mean_absolute_error(y_test, y_pred))
print("Baseline RMSE:", root_mean_squared_error(y_test, y_pred))


Dataset shape: (4981, 17)
Total rows: 4981
Unique patients: 100
Target rows with zero Insulin_Dose: 800
Target unique values: 6
Dataset size verdict: Good for ensemble models


,Patient_ID,Timestamp,Glucose_Level,Heart_Rate,Activity_Level,Calories_Burned,Sleep_Duration,Step_Count,Insulin_Dose,Medication_Intake,Diet_Quality_Score,Stress_Level,BMI,HbA1c,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Predicted_Progression
0,P052,2025-06-01 00:00:00,165.7,67.0,68,193.1,4.4,8832,8,1,8,4,26.2,5.5,109,68,1
1,P093,2025-06-01 00:15:00,114.0,70.0,38,205.9,4.3,2255,0,0,5,9,24.5,6.2,152,79,1
2,P015,2025-06-01 00:30:00,125.2,81.0,69,274.5,6.1,1299,4,1,8,9,26.0,5.5,143,99,2
3,P072,2025-06-01 00:45:00,157.1,82.0,99,141.9,4.2,8809,10,1,4,9,22.1,6.8,112,60,0
4,P061,2025-06-01 01:00:00,156.8,62.0,3,245.7,7.7,5521,10,0,10,3,21.4,7.7,152,72,1



Missing values per column:


Patient_ID                  0
Timestamp                   0
Glucose_Level               0
Heart_Rate                  0
Activity_Level              0
Calories_Burned             0
Sleep_Duration              0
Step_Count                  0
Insulin_Dose                0
Medication_Intake           0
Diet_Quality_Score          0
Stress_Level                0
BMI                         0
HbA1c                       0
Blood_Pressure_Systolic     0
Blood_Pressure_Diastolic    0
Predicted_Progression       0
dtype: int64


Numeric summary statistics:


...

C:\Users\USER\AppData\Local\Temp\ipykernel_13160\3922926406.py:183: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\USER\AppData\Local\Temp\ipykernel_13160\3922926406.py:185: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = raw_df.select_dtypes(include=["object"]).columns.tolist()


day_of_week
Sunday       768
Monday       768
Tuesday      757
Wednesday    672
Thursday     672
Friday       672
Saturday     672
Name: count, dtype: int64


month (2 unique values)


month
June    2880
July    2101
Name: count, dtype: int64


Baseline linear regression check:
Skipping missing engineered features: ['glycemic_stress_index', 'pulse_pressure', 'activity_volume']
Baseline train rows: 4038
Baseline test rows: 943
Baseline Linear Regression R2: -0.003929977759876246
Baseline MAE: 2.989296255978685
Baseline RMSE: 3.398523140451197


## Preprocessing and Training Pipeline

This section shows the data preprocessing flow from raw CSV loading through feature engineering, group-aware splitting, and first baseline model training.

- Load the raw SmartSensor dataset
- Prepare the modeling frame with derived clinical and timestamp features
- Apply patient-grouped train/test splitting
- Fit a baseline regression pipeline and report metrics

In [ ]:
from clinical_insulin_pipeline.data.dataset import prepare_dataset
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

bundle = prepare_dataset(csv_path, apply_iqr=True)
print("Prepared dataset:")
print("  X_train shape:", bundle.X_train.shape)
print("  X_test shape:", bundle.X_test.shape)
print("  Rows removed by IQR filtering:", bundle.n_rows_dropped_iqr)
print("  Feature count:", len(bundle.feature_names))

model_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]
)

model_pipe.fit(bundle.X_train, bundle.y_train)
y_pred = model_pipe.predict(bundle.X_test)

print("\nBaseline training flow results:")
print("  Train rows:", len(bundle.X_train))
print("  Test rows:", len(bundle.X_test))
print("  R2:", r2_score(bundle.y_test, y_pred))
print("  MAE:", mean_absolute_error(bundle.y_test, y_pred))
print("  RMSE:", np.sqrt(mean_squared_error(bundle.y_test, y_pred)))
print("  Sample feature names:", bundle.feature_names[:10])

## Data Cleaning, Feature Engineering, and Split Validation

This section performs the core preprocessing steps that are critical for regression modeling:

- Imputation for missing values (median for numeric, mode/"Missing" for categorical)
- Timestamp transformation into cyclical features
- Domain-derived features from clinical signals
- Outlier detection using box plots and z-scores
- Correlation and VIF analysis to identify redundant or low-value features
- Group-aware train/test split so the same patient does not appear in both sets

In [50]:
from clinical_insulin_pipeline.data.dataset import load_raw_csv, prepare_dataset, build_modeling_frame
from clinical_insulin_pipeline.data.features import (
    add_cyclical_time_features,
    add_derived_clinical_features,
    feature_columns_after_engineering,
)
from clinical_insulin_pipeline.config import TARGET_COL
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

raw_df = load_raw_csv(csv_path)
print("Raw CSV shape:", raw_df.shape)
display(raw_df.head())

print("\nMissing values per column:")
display(raw_df.isna().sum().sort_values(ascending=False))

print("\nData types:")
display(raw_df.dtypes)

numeric_cols = raw_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = raw_df.select_dtypes(include=["object"]).columns.difference(["Timestamp"]).tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical/object columns:", categorical_cols)

clean_df = raw_df.copy()
for col in numeric_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in categorical_cols:
    if clean_df[col].isna().any():
        mode_value = clean_df[col].mode().iloc[0] if not clean_df[col].mode().empty else "Missing"
        clean_df[col] = clean_df[col].fillna(mode_value)

print("\nAfter imputation, missing values remaining:")
display(clean_df.isna().sum().sort_values(ascending=False))

clean_df = add_cyclical_time_features(clean_df)
clean_df = add_derived_clinical_features(clean_df)

feature_df = clean_df[feature_columns_after_engineering()]
print("\nFeatures after engineering:", feature_df.shape[1])
display(feature_df.head())

corr_with_target = feature_df.join(clean_df[TARGET_COL]).corr()[TARGET_COL].abs().sort_values(ascending=False)
print("\nFeature correlation with target:")
display(corr_with_target)

def compute_vif(X: pd.DataFrame) -> pd.Series:
    vif_values = {}
    for col in X.columns:
        y = X[col].values.reshape(-1, 1)
        X_other = X.drop(columns=[col]).values
        model = LinearRegression().fit(X_other, y)
        r2 = model.score(X_other, y)
        vif_values[col] = float("inf") if np.isclose(r2, 1.0) else 1.0 / (1.0 - r2)
    return pd.Series(vif_values).sort_values(ascending=False)

print("\nTop VIF scores (multicollinearity):")
display(compute_vif(feature_df).head(15))

outlier_cols = ["BMI", "Blood_Pressure_Systolic", "Blood_Pressure_Diastolic"]
fig, ax = plt.subplots(figsize=(10, 5))
clean_df[outlier_cols].boxplot(ax=ax)
ax.set_title("Outlier candidates for BMI and blood pressure")
plt.show()

z_scores = np.abs((feature_df - feature_df.mean()) / feature_df.std(ddof=0))
high_z = (z_scores > 3).sum().sort_values(ascending=False)
print("\nCount of extreme z-score values by feature:")
display(high_z[high_z > 0])

bundle = prepare_dataset(csv_path)
print("\nGroup-aware train/test split:")
print("  Train rows:", len(bundle.X_train))
print("  Test rows:", len(bundle.X_test))
print("  Patient groups in train/test:", len(set(bundle.groups_train)), len(set(bundle.groups_test)))
print("  Rows removed by IQR outlier filtering:", bundle.n_rows_dropped_iqr)
print("  Final modeling features:", len(bundle.feature_names))


Raw CSV shape: (4981, 17)


,Patient_ID,Timestamp,Glucose_Level,Heart_Rate,Activity_Level,Calories_Burned,Sleep_Duration,Step_Count,Insulin_Dose,Medication_Intake,Diet_Quality_Score,Stress_Level,BMI,HbA1c,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Predicted_Progression
0,P052,2025-06-01 00:00:00,165.7,67.0,68,193.1,4.4,8832,8,1,8,4,26.2,5.5,109,68,1
1,P093,2025-06-01 00:15:00,114.0,70.0,38,205.9,4.3,2255,0,0,5,9,24.5,6.2,152,79,1
2,P015,2025-06-01 00:30:00,125.2,81.0,69,274.5,6.1,1299,4,1,8,9,26.0,5.5,143,99,2
3,P072,2025-06-01 00:45:00,157.1,82.0,99,141.9,4.2,8809,10,1,4,9,22.1,6.8,112,60,0
4,P061,2025-06-01 01:00:00,156.8,62.0,3,245.7,7.7,5521,10,0,10,3,21.4,7.7,152,72,1



Missing values per column:


Patient_ID                  0
Timestamp                   0
Glucose_Level               0
Heart_Rate                  0
Activity_Level              0
Calories_Burned             0
Sleep_Duration              0
Step_Count                  0
Insulin_Dose                0
Medication_Intake           0
Diet_Quality_Score          0
Stress_Level                0
BMI                         0
HbA1c                       0
Blood_Pressure_Systolic     0
Blood_Pressure_Diastolic    0
Predicted_Progression       0
dtype: int64


Data types:


...

month_cos                          inf
month_sin                          inf
pulse_pressure                     inf
Blood_Pressure_Systolic            inf
Blood_Pressure_Diastolic           inf
activity_volume             174.494936
Activity_Level              172.235318
glycemic_stress_index        66.367906
Glucose_Level                44.802178
HbA1c                        23.506460
Step_Count                    3.226704
Stress_Level                  1.005803
dow_cos                       1.004671
Heart_Rate                    1.004486
Diet_Quality_Score            1.004470
dtype: float64


Count of extreme z-score values by feature:


C:\Users\USER\AppData\Local\Temp\ipykernel_13160\3107468713.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


glycemic_stress_index    21
Calories_Burned          18
BMI                      18
HbA1c                    13
Heart_Rate               12
Glucose_Level            11
dtype: int64


Group-aware train/test split:
  Train rows: 4004
  Test rows: 939
  Patient groups in train/test: 80 20
  Rows removed by IQR outlier filtering: 38
  Final modeling features: 22


In [51]:
# Train candidate models, evaluate, select best by test RMSE, and deploy the bundle
res = run_training(
    csv_path,
    skip_learning_curve=SKIP_LEARNING_CURVE,
    skip_shap=SKIP_SHAP,
)

print("Best model selected:", res.best_name)
print("Test metrics:", res.test_metrics)
print("Output artifacts directory:", res.output_dir)
print("Best bundle path in training output:", res.output_dir / "inference_bundle.joblib")
print("\nLeaderboard:")
display(res.leaderboard)
print("\nPer-patient RMSE for the best model:")
display(res.per_patient_rmse.sort_values("rmse").head(10))

from insulin_system.persistence.bundle import resolve_inference_bundle_path
from insulin_system.persistence import load_best_model

backend_bundle_path = resolve_inference_bundle_path(None)
print("\nBackend deployed bundle path:", backend_bundle_path)
print("Backend bundle exists:", backend_bundle_path.is_file())

print("\nLoad the deployed bundle through the app persistence layer:")
bundle = load_best_model()
print("Loaded bundle path:", bundle.path)
print("Loaded model name:", bundle.model_name)
print("Loaded feature count:", len(bundle.feature_names))
print("Loaded test metrics:", bundle.data.get("test_metrics"))


INFO random_forest test RMSE=3.4271 MAE=2.9882 R2=-0.0201
INFO hist_gradient_boosting test RMSE=3.4177 MAE=2.9955 R2=-0.0146
INFO xgboost test RMSE=3.5816 MAE=3.0498 R2=-0.1142
INFO Saved evaluation CSVs under E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation
INFO Deployed bundle to E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib


Best model selected: hist_gradient_boosting
Test metrics: {'mae': 2.995465930226316, 'rmse': 3.4177145401316573, 'mape': 73473650.93316962, 'r2': -0.014555452148053982, 'max_error': 5.980205492800214}
Output artifacts directory: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest
Best bundle path in training output: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\inference_bundle.joblib

Leaderboard:


,model,mae,rmse,mape,r2,max_error
0,hist_gradient_boosting,2.995466,3.417715,7.347365e+07,-0.014555,5.980205
1,random_forest,2.988239,3.427054,7.342717e+07,-0.020108,6.212742
2,xgboost,3.049848,3.581590,7.443707e+07,-0.114181,8.039640



Per-patient RMSE for the best model:


,patient,mae,sq_err,rmse
15,P077,2.474273,9.135348,3.022474
2,P011,2.763511,9.934502,3.151905
12,P054,2.787239,10.111250,3.179819
6,P031,2.809539,10.464940,3.234956
0,P001,2.891603,11.004012,3.317230
18,P084,2.938235,11.090878,3.330297
4,P019,2.876991,11.292580,3.360443
14,P074,2.991182,11.306217,3.362472
19,P091,2.936893,11.325825,3.365386
7,P032,2.997987,11.561295,3.400190


INFO Loaded clinical insulin bundle from E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib



Backend deployed bundle path: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Backend bundle exists: True

Load the deployed bundle through the app persistence layer:
Loaded bundle path: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Loaded model name: hist_gradient_boosting
Loaded feature count: 22
Loaded test metrics: {'mae': 2.995465930226316, 'rmse': 3.4177145401316573, 'mape': 73473650.93316962, 'r2': -0.014555452148053982, 'max_error': 5.980205492800214}


In [52]:
# Where artifacts were saved
from insulin_system.persistence.bundle import resolve_inference_bundle_path

latest_dir = res.output_dir
bundle_file = latest_dir / "inference_bundle.joblib"

print("Run output dir:", latest_dir)
print("Leaderboard:", latest_dir / "leaderboard.csv")
print("Eval metrics:", latest_dir / "evaluation" / "model_metrics.csv")
print("Eval summary:", latest_dir / "evaluation" / "evaluation_summary.csv")
print("Plots dir:", latest_dir / "models")
print("Bundle saved in training output:", bundle_file)
print("Saved bundle exists:", bundle_file.is_file())

backend_bundle_path = resolve_inference_bundle_path(None)
print("Backend deployed bundle path:", backend_bundle_path)
print("Backend bundle exists:", backend_bundle_path.is_file())


Run output dir: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest
Leaderboard: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\leaderboard.csv
Eval metrics: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation\model_metrics.csv
Eval summary: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation\evaluation_summary.csv
Plots dir: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\models
Bundle saved in training output: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\inference_bundle.joblib
Saved bundle exists: False
Backend deployed bundle path: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Backend bundle exists: True


In [53]:
# Verify the backend can load the saved bundle
from insulin_system.persistence import load_best_model

bundle = load_best_model()
print("Loaded bundle:", bundle.path)
print("Model name:", bundle.model_name)
print("n_features:", len(bundle.feature_names))
print("Test metrics in bundle:", bundle.data.get("test_metrics"))


INFO Loaded clinical insulin bundle from E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib


Loaded bundle: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Model name: hist_gradient_boosting
n_features: 22
Test metrics in bundle: {'mae': 2.995465930226316, 'rmse': 3.4177145401316573, 'mape': 73473650.93316962, 'r2': -0.014555452148053982, 'max_error': 5.980205492800214}


## End-to-End Prediction from Loaded Bundle
This cell demonstrates the full flow from trained model bundle loading to final insulin dose prediction using the app-facing inference schema.

In [54]:
from clinical_insulin_pipeline.serving.predict import predict_from_insulin_prediction_input
from clinical_insulin_pipeline.serving.predict import predict_from_insulin_prediction_input
from clinical_insulin_pipeline.serving.schema import InsulinPredictionInput
from insulin_system.persistence import load_best_model

sample_input = InsulinPredictionInput(
    timestamp=None,
    glucose_level=180.0,
    heart_rate=78.0,
    activity_level=30.0,
    calories_burned=140.0,
    sleep_duration=6.5,
    step_count=3200.0,
    medication_intake=0,
    diet_quality_score=5.0,
    stress_level=7.0,
    bmi=28.0,
    hba1c=7.2,
    blood_pressure_systolic=132.0,
    blood_pressure_diastolic=85.0,
)

loaded_bundle = load_best_model()
final_dose = predict_from_insulin_prediction_input(loaded_bundle.data, sample_input)

print("Sample input:", sample_input)
print("Final predicted insulin dose (IU):", final_dose)

INFO Loaded clinical insulin bundle from E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib


Sample input: InsulinPredictionInput(timestamp=None, age_years=None, bmi=28.0, hba1c=7.2, blood_pressure_systolic=132.0, blood_pressure_diastolic=85.0, glucose_level=180.0, heart_rate=78.0, activity_level=30.0, calories_burned=140.0, sleep_duration=6.5, step_count=3200.0, diet_quality_score=5.0, stress_level=7.0, medication_intake=0, extra={})
Final predicted insulin dose (IU): 5.5


## Notes

- If `CSV exists` is `False`, place your dataset at `Clinical-Insulin-Recommendation/data/SmartSensor_DiabetesMonitoring.csv` (or change `csv_path`).
- This notebook now performs explicit data cleaning, outlier detection, cyclical timestamp encoding, derived clinical features, multicollinearity checks, and group-aware train/test splitting.
- The training pipeline evaluates multiple candidate regression models, chooses the best model by test RMSE, saves the best bundle, and deploys it to `outputs/best_model/inference_bundle.joblib`.
- If you want a faster run, set:
  - `SKIP_LEARNING_CURVE = True`
  - `SKIP_SHAP = True`

After training, start the backend and the UI:

```bash
# from Clinical-Insulin-Recommendation/
python -m uvicorn backend.app:app --reload --port 8000

# in another terminal
cd frontend
npm run dev
```
